In [2]:
import duckdb
import sys
from pathlib import Path
import pandas as pd 

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
    
    
from src.db.sql_runner import run_query



# Antes de começar a Inteligência de Negócio, validar o dataset e realizar limpeza de acordo com as regras de negócio

In [ ]:
df_nulos = run_query("01_nulls_check.sql")
df_nulos

In [4]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()

colunas_mantidas = df_nulos[df_nulos['pct_nulos'] <= 85]['column_name'].tolist()

print(f'Total de {len(df_nulos)} colunas originais')
print(f'Total de {len(colunas_descarte)} colunas para descarte (>85% nulos)')
print(f'Total de {len(colunas_mantidas)} colunas mantidas (<=85% nulos)')

NameError: name 'df_nulos' is not defined

In [ ]:
df_emails = run_query("02_ordenar_emails.sql")
df_emails

In [ ]:
agrupar_emails_comprador = run_query('03_agrupar_emails_comprador.sql')
agrupar_emails_comprador

In [ ]:
agrupar_emails_destino = run_query('04_agrupar_emails_destino.sql')
agrupar_emails_destino

In [ ]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()
colunas_descarte_str = ', '.join(colunas_descarte)
colunas_descarte_str

In [ ]:
view_limpa = run_query("05_view_limpa.sql", colunas_descarte_str=colunas_descarte_str)
view_limpa

In [ ]:
query_limpa = run_query('06_test_view.sql')
query_limpa

In [ ]:
duckdb.execute(r"COPY vw_train_clean TO 'caminho para o arquivo' (FORMAT PARQUET)")

# Documentação: Tratamento de Dados e Engenharia de Features (Camada Silver)
Objetivo: Registrar o pipeline de limpeza, governança de nulos e normalização de categorias aplicados no dataset de detecção de fraudes para a consolidação da camada Silver (train_clean.parquet).
# 1. Regras de Governança e Qualidade de Dados (DQ)
Regra: Identificação e descarte de colunas que apresentem taxa de valores ausentes (nulos) superior a 85%.
Motivação: Variáveis com densidade de dados inferior a 15% introduzem ruído ao modelo e reduzem a eficiência computacional, sem agregar sinal preditivo relevante.
Resultado: 74 colunas excederam o limite do SLA e foram removidas dinamicamente do dataset através da cláusula EXCLUDE do DuckDB.
# 2. Engenharia de Features e Categorização (ID)
### Normalização de Domínios de E-mail (ID-02)
As colunas originais de e-mail do comprador (P_emaildomain) e do destinatário (R_emaildomain) apresentavam alta cardinalidade e fragmentação (ex: variações como gmail.com, gmail, hotmail.com, hotmail.co.uk).
Regra Aplicada: Mapeamento via sintaxe condicional CASE WHEN agrupando os domínios em 7 categorias estratégicas:

google

microsoft

yahoo

apple

anonymous

missing (preservação explícita de registros nulos para análise de risco)

outro (provedores corporativos/raros)
# Novas Features Geradas:
categoria_provedor_comprador: Provedor tratado do e-mail de quem realiza a compra.

categoria_provedor_destino: Provedor tratado do e-mail do destinatário (sinal valioso para fraudes em gift cards e entregas a terceiros).
# 4. Pipeline de Materialização e Exportação
Construção da View Virtual: As regras de negócio foram unificadas em uma VIEW lógica no DuckDB (vw_train_clean), evitando duplicação de dados na memória RAM durante os testes.

Exportação de Alta Performance: O arquivo final foi persistido diretamente do motor C++ do DuckDB para o disco, eliminando gargalos do PyArrow e otimizando o tempo de gravação.
Caminho de Destino: data/processed/train_clean.parquet
Tempo de Execução: 10,8 segundos para gravação de 590.540 linhas x 362 colunas.

# Pulando de camada (Bronze para Silver)
PARQUET_PATH do sql_runner alterado para parquet_limpo

In [5]:
risco_provedor_comprador = run_query('07_calculo_fraude_email_comp.sql')
risco_provedor_comprador

,categoria_provedor_comprador,total_transacoes_email_comprador,total_fraudes_email_comprador,porcentagem_fraude_email_comprador
0,microsoft,59477,3170.0,5.33
1,google,228851,9954.0,4.35
2,missing,94456,2790.0,2.95
3,apple,8225,238.0,2.89
4,anonymous,36998,859.0,2.32
5,outro,56564,1280.0,2.26
6,yahoo,105969,2372.0,2.24


In [6]:
risco_provedor_destino = run_query('08_calculo_fraude_email_dest.sql')
risco_provedor_destino

,categoria_provedor_destino,total_transacoes_email_destino,total_fraudes_email_destino,porcentagem_fraude_email_destino
0,google,57242,6811.0,11.90
1,apple,2172,193.0,8.89
2,microsoft,33604,2714.0,8.08
3,yahoo,13967,644.0,4.61
4,anonymous,20529,598.0,2.91
5,outro,9777,267.0,2.73
6,missing,453249,9436.0,2.08
